# Feature Engineering & Preprocessing — ATD Dataset

This notebook transforms the raw delivery data into a clean, feature-rich dataset ready for predictive modeling and business analysis.

## Objectives
1. **Validate** the ATD formula and confirm timestamp alignment
2. **Engineer** time-based, distance, and wait-time features
3. **Handle** missing values with statistically justified imputation
4. **Treat outliers** in ATD and distance columns
5. **Encode** categorical features for downstream modeling
6. **Save** the processed dataset to `data/processed/features.parquet`

---
**Sections**
1. Setup & Data Load
2. ATD Formula Validation
3. Time Feature Engineering
4. Wait-Time / Preparation Window Features
5. Distance Feature Engineering
6. Missing Value Imputation
7. Outlier Treatment
8. Categorical Encoding
9. Feature Summary & Save

---
## 1. Setup & Data Load

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})
SEED = 42

RAW_PATH      = Path('../data/raw/BC_A&A_with_ATD.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_PATH = PROCESSED_DIR / 'features.parquet'

# UTC offset fallback used only when a territory cannot be mapped
_DEFAULT_OFFSET_H = 6

In [ ]:
dtype_map = {
    'region': 'category', 'territory': 'category',
    'country_name': 'category', 'courier_flow': 'category',
    'geo_archetype': 'category', 'merchant_surface': 'category',
}

df = pd.read_csv(
    RAW_PATH,
    dtype=dtype_map,
    parse_dates=[
        'restaurant_offered_timestamp_utc',
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
    ],
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
print(f'Date range (UTC): {df["restaurant_offered_timestamp_utc"].min()} → {df["restaurant_offered_timestamp_utc"].max()}')
df.head(3)

In [ ]:
# ── Territory → UTC offset mapping (data-driven) ────────────────────────────
#
# Rather than hardcoding UTC-6, we infer the correct offset for each territory
# directly from the data.
#
# Formula (rearranged from ATD definition):
#   ATD = (local_final + offset_h − utc_offered).total_seconds / 60
#   ⟹  offset_h = ATD/60 − (local_final − utc_offered).total_seconds / 3600
#
# We use the pre-existing ATD column (which was computed with the correct per-row
# timezone at ingestion) to back-solve the offset, then take the mode per territory.

_clean = (
    df['ATD'].notna() & (df['ATD'] > 0) & (df['ATD'] < 240) &
    df['order_final_state_timestamp_local'].notna() &
    df['restaurant_offered_timestamp_utc'].notna()
)

_naive_diff_h = (
    df.loc[_clean, 'order_final_state_timestamp_local']
    - df.loc[_clean, 'restaurant_offered_timestamp_utc']
).dt.total_seconds() / 3600

_implied_offset = (df.loc[_clean, 'ATD'] / 60) - _naive_diff_h

# Round to nearest integer hour and take the mode per territory
territory_tz = (
    pd.DataFrame({
        'territory': df.loc[_clean, 'territory'].values,
        'offset'   : _implied_offset.round().astype(int).values,
    })
    .groupby('territory')['offset']
    .agg(lambda x: int(x.mode()[0]))
    .rename('utc_offset_h')
)

# Map to every row; fall back to _DEFAULT_OFFSET_H for any unmapped territory
df['utc_offset_h'] = df['territory'].map(territory_tz).fillna(_DEFAULT_OFFSET_H).astype(int)

# Single vectorised timedelta Series reused in all subsequent timestamp arithmetic
tz_offset = pd.to_timedelta(df['utc_offset_h'], unit='h')

print('Territory → UTC offset (hours to ADD to local time to obtain UTC):')
display(territory_tz.sort_values().to_frame())

print('\nutc_offset_h distribution across all rows:')
display(
    df['utc_offset_h'].value_counts()
    .sort_index()
    .rename('rows')
    .to_frame()
    .assign(pct=lambda x: (x['rows'] / len(df) * 100).round(2))
)

---
## 2. ATD Formula Validation

**ATD** is defined as the elapsed time (minutes) between when the restaurant received the order and when the order reached its final state.

> `ATD = (order_final_state_timestamp_local − restaurant_offered_timestamp_utc + 6 h) / 60`

Since `order_final_state_timestamp_local` is Mexico local time (UTC−6), we add 6 hours to convert it to UTC before differencing.

In [ ]:
# Recompute ATD using per-row timezone offset (vectorised)
df['atd_computed'] = (
    (df['order_final_state_timestamp_local'] + tz_offset
     - df['restaurant_offered_timestamp_utc'])
    .dt.total_seconds() / 60
)

diff = (df['ATD'] - df['atd_computed']).dropna()
print('ATD residual = provided − computed (per-territory offset):')
print(f'  rows evaluated : {diff.notna().sum():,}')
print(f'  mean           : {diff.mean():.6f} min')
print(f'  std            : {diff.std():.6f} min')
print(f'  max |residual| : {diff.abs().max():.6f} min')
print(f'  |residual|<0.1 : {(diff.abs() < 0.1).mean()*100:.3f}%')
print(f'  |residual|>1   : {(diff.abs() > 1).sum():,} rows')

# Residual breakdown by territory to spot any remaining mismatches
res_by_territory = (
    df.assign(_res=diff.abs())
    .groupby('territory', observed=True)['_res']
    .agg(['mean', 'max', lambda x: (x > 1).mean() * 100])
    .rename(columns={'mean': 'mean_abs_res', 'max': 'max_abs_res', '<lambda_0>': 'pct_>1min'})
    .sort_values('mean_abs_res', ascending=False)
)
print('\nResidual by territory (should be near-zero everywhere):')
display(res_by_territory.round(4))

sample_row = df[df['ATD'].notna() & df['atd_computed'].notna()].iloc[0]
print(f'\nExample row (territory={sample_row["territory"]}, offset={sample_row["utc_offset_h"]}h):')
print(f'  restaurant_offered_utc  : {sample_row["restaurant_offered_timestamp_utc"]}')
print(f'  order_final_state_local : {sample_row["order_final_state_timestamp_local"]}')
print(f'  ATD (provided)          : {sample_row["ATD"]:.4f} min')
print(f'  ATD (computed)          : {sample_row["atd_computed"]:.4f} min')

In [ ]:
# Use the provided ATD column (validated above); drop the computed helper
df.drop(columns=['atd_computed'], inplace=True)
print('ATD column validated. Using provided ATD for all downstream work.')

---
## 3. Time Feature Engineering

Time-based features are critical because delivery demand and network congestion vary strongly by hour, day, and week.

In [ ]:
ts = df['restaurant_offered_timestamp_utc']
_original_cols = df.columns.tolist()  # snapshot before adding features

# UTC-based features
df['hour_utc']        = ts.dt.hour
df['day_of_week']     = ts.dt.dayofweek          # 0=Mon, 6=Sun
df['day_name']        = ts.dt.day_name()
# isocalendar() returns UInt32 with pd.NA for NaT rows; use nullable 'Int32' to avoid cast error
df['week_number']     = ts.dt.isocalendar().week.astype('Int32')
df['month']           = ts.dt.month
df['day_of_month']    = ts.dt.day

# Local hour using per-territory UTC offset (not a hardcoded constant)
df['hour_local']      = (df['hour_utc'] - df['utc_offset_h']).mod(24)

# Weekend flag
df['is_weekend']      = df['day_of_week'].isin([5, 6]).astype(int)

# Peak-hour flag (local time): lunch 12–15h, dinner/late 19–23h
df['is_peak_hour']    = df['hour_local'].isin(list(range(12, 15)) + list(range(19, 24))).astype(int)

# Time-of-day block (local) — vectorised np.select (~30× faster than .apply on 1 M rows)
h = df['hour_local']
df['time_block'] = pd.Categorical(
    np.select(
        condlist=[
            (h >= 6)  & (h < 11),
            (h >= 11) & (h < 14),
            (h >= 14) & (h < 18),
            (h >= 18) & (h < 22),
            (h >= 22),
        ],
        choicelist=['morning', 'lunch', 'afternoon', 'dinner', 'late_night'],
        default='overnight',
    )
)

# Day type — np.where returns a NumPy array; cast after column assignment
df['day_type'] = np.where(
    df['is_weekend'],
    'weekend',
    np.where(df['hour_local'] < 12, 'weekday_am', 'weekday_pm'),
)
df['day_type'] = df['day_type'].astype('category')

print('Time features created:')
print([c for c in df.columns if c not in _original_cols])
df[['restaurant_offered_timestamp_utc', 'hour_utc', 'utc_offset_h', 'hour_local',
    'day_of_week', 'is_weekend', 'is_peak_hour', 'time_block', 'day_type']].head(5)

In [ ]:
DOW = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BLOCK_ORDER = ['overnight', 'morning', 'lunch', 'afternoon', 'dinner', 'late_night']

fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Volume by local hour
df.groupby('hour_local').size().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Trip Volume by Local Hour')
axes[0].set_xlabel('Local Hour (UTC−6)')
axes[0].tick_params(axis='x', rotation=0)

# Median ATD by local hour
df.groupby('hour_local')['ATD'].median().plot(
    kind='line', ax=axes[1], color='coral', marker='o', linewidth=2)
axes[1].set_title('Median ATD by Local Hour')
axes[1].set_xlabel('Local Hour (UTC−6)')
axes[1].set_ylabel('Median ATD (min)')

# Median ATD by time block
tb_atd = df.groupby('time_block', observed=True)['ATD'].median().reindex(BLOCK_ORDER)
tb_atd.plot(kind='bar', ax=axes[2], color='teal', edgecolor='none')
axes[2].set_title('Median ATD by Time Block')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Time Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Wait-Time / Preparation Window Features

The time between the **eater's request** and the **restaurant being offered** the order captures:
- Platform matching latency
- Time until a courier was dispatched

This is a proxy for courier availability / dispatch efficiency.

In [ ]:
# Eater wait: from customer request (local) to restaurant offer (UTC)
# Convert request to UTC using per-territory offset, then difference
df['eater_wait_min'] = (
    (df['restaurant_offered_timestamp_utc']
     - (df['eater_request_timestamp_local'] + tz_offset))
    .dt.total_seconds() / 60
)

print('eater_wait_min stats:')
print(df['eater_wait_min'].describe().round(2))
print(f'\nNegative eater_wait_min : {(df["eater_wait_min"] < 0).sum():,}')
print(f'>60 min eater_wait_min  : {(df["eater_wait_min"] > 60).sum():,}')

# Breakdown by territory — should all cluster near 0 now that offsets are correct
print('\nMedian eater_wait_min by territory (should be small positive everywhere):')
display(
    df.groupby('territory', observed=True)['eater_wait_min']
    .median()
    .sort_values(ascending=False)
    .round(2)
    .to_frame()
)

In [ ]:
# Clip eater_wait_min to [0, 60] — values outside range indicate data quality issues
df['eater_wait_min'] = df['eater_wait_min'].clip(lower=0, upper=60)

# Bin into categories
wait_bins   = [0, 2, 5, 10, 20, 60]
wait_labels = ['0-2', '2-5', '5-10', '10-20', '20-60']
df['eater_wait_bin'] = pd.cut(
    df['eater_wait_min'], bins=wait_bins, labels=wait_labels, include_lowest=True
).astype('category')

# Check relationship with ATD
wait_atd = df.groupby('eater_wait_bin', observed=True)['ATD'].agg(['median', 'count']).round(1)
print('Median ATD by eater wait bin:')
display(wait_atd)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['eater_wait_min'].dropna(), bins=60, color='mediumpurple', edgecolor='none')
axes[0].set_title('Distribution of Eater Wait Time (clipped 0–60 min)')
axes[0].set_xlabel('Minutes')

corr = df[['eater_wait_min', 'ATD']].dropna().corr().iloc[0, 1]
sample = df[['eater_wait_min', 'ATD']].dropna().sample(20_000, random_state=SEED)
axes[1].scatter(sample['eater_wait_min'], sample['ATD'], alpha=0.04, s=4, color='mediumpurple')
axes[1].set_title(f'Eater Wait vs ATD  (Pearson r={corr:.3f})')
axes[1].set_xlabel('Eater Wait (min)')
axes[1].set_ylabel('ATD (min)')

plt.suptitle('Eater Wait-Time Feature', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Distance Feature Engineering

Distance is the strongest numeric predictor of ATD. We create multiple representations to capture linear and non-linear effects.

In [ ]:
# Derived distance features
df['total_distance']   = df['pickup_distance'] + df['dropoff_distance']
df['distance_ratio']   = df['dropoff_distance'] / (df['pickup_distance'] + 0.001)  # avoid div/0
df['is_long_trip']     = (df['total_distance'] > 5).astype(int)  # > 5 km total

# Log transforms (useful for right-skewed features)
df['log_pickup']       = np.log1p(df['pickup_distance'])
df['log_dropoff']      = np.log1p(df['dropoff_distance'])
df['log_total_dist']   = np.log1p(df['total_distance'])

# Binned total distance
dist_bins   = [0, 1, 2, 3, 5, 8, 15, np.inf]
dist_labels = ['0-1', '1-2', '2-3', '3-5', '5-8', '8-15', '15+']
df['total_dist_bin'] = pd.cut(
    df['total_distance'], bins=dist_bins, labels=dist_labels, include_lowest=True
).astype('category')

print('Distance features created.')
print('\nCorrelation with ATD:')
dist_cols = ['pickup_distance', 'dropoff_distance', 'total_distance',
             'distance_ratio', 'log_pickup', 'log_dropoff', 'log_total_dist']
corr_atd = df[dist_cols + ['ATD']].dropna().corr()['ATD'].drop('ATD').sort_values(ascending=False)
display(corr_atd.round(4).to_frame('pearson_r'))

In [ ]:
p99_atd  = df['ATD'].quantile(0.99)
p99_dist = df['total_distance'].quantile(0.99)

df_plot = df[
    (df['ATD'] <= p99_atd) &
    (df['total_distance'] <= p99_dist) &
    df['total_distance'].notna()
].sample(25_000, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].scatter(df_plot['total_distance'], df_plot['ATD'], alpha=0.03, s=4, color='teal')
axes[0].set(title='Total Distance vs ATD', xlabel='Total Distance (km)', ylabel='ATD (min)')

axes[1].scatter(df_plot['log_total_dist'], df_plot['ATD'], alpha=0.03, s=4, color='steelblue')
axes[1].set(title='log(Total Distance) vs ATD', xlabel='log1p(total_distance)', ylabel='ATD (min)')

atd_by_bin = df.groupby('total_dist_bin', observed=True)['ATD'].median()
atd_by_bin.plot(kind='bar', ax=axes[2], color='coral', edgecolor='none')
axes[2].set(title='Median ATD by Distance Bin', xlabel='Total Distance (km)', ylabel='Median ATD (min)')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Distance Features vs ATD', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Missing Value Imputation

~2.5% of rows are missing both `pickup_distance` and `dropoff_distance`. We impute using the **territory × courier_flow × time_block** median, then fall back to territory-level median, then global median.

In [ ]:
dist_cols_raw = ['pickup_distance', 'dropoff_distance']
missing_counts = df[dist_cols_raw + ['eater_wait_min']].isnull().sum()
print('Missing values before imputation:')
display(missing_counts.to_frame('count').assign(pct=lambda x: (x['count'] / len(df) * 100).round(2)))

# Flag for models: knowing a distance was missing can itself be predictive
df['is_distance_missing'] = df['pickup_distance'].isna().astype(int)

In [ ]:
def hierarchical_median_impute(series: pd.Series, *group_cols) -> pd.Series:
    """Impute missing values using hierarchical group medians."""
    result = series.copy()
    mask   = result.isna()
    if mask.sum() == 0:
        return result

    for i in range(len(group_cols), 0, -1):
        cols = list(group_cols[:i])
        grp_median = df.groupby(cols, observed=True)[series.name].transform('median')
        still_missing = result.isna()
        result[still_missing] = grp_median[still_missing]

    # Final fallback: global median
    result.fillna(result.median(), inplace=True)
    return result


for col in dist_cols_raw:
    df[col] = hierarchical_median_impute(
        df[col], 'territory', 'courier_flow', 'time_block'
    )

# Recompute derived distance features after imputation
df['total_distance']   = df['pickup_distance'] + df['dropoff_distance']
df['distance_ratio']   = df['dropoff_distance'] / (df['pickup_distance'] + 0.001)
df['log_pickup']       = np.log1p(df['pickup_distance'])
df['log_dropoff']      = np.log1p(df['dropoff_distance'])
df['log_total_dist']   = np.log1p(df['total_distance'])
df['total_dist_bin']   = pd.cut(
    df['total_distance'], bins=dist_bins, labels=dist_labels, include_lowest=True
).astype('category')

# Impute eater_wait_min with territory × time_block median
df['eater_wait_min'] = hierarchical_median_impute(df['eater_wait_min'], 'territory', 'time_block')

print('Missing values after imputation:')
display(df[dist_cols_raw + ['eater_wait_min']].isnull().sum().to_frame('count'))

---
## 7. Outlier Treatment

Extreme ATD values (< 1 min or > 120 min) are either data quality issues or edge cases that would distort model training. We apply a two-step strategy:
- **Remove** impossible values (ATD ≤ 0)
- **Cap (Winsorise)** at the 99th percentile per segment for modeling purposes
- **Flag** the originals for separate outlier analysis

In [ ]:
print('ATD outlier profile:')
for lo, hi, label in [
    (None, 0,   'ATD ≤ 0 (impossible)'),
    (0, 1,      'ATD in (0, 1) min'),
    (120, None, 'ATD > 120 min'),
    (240, None, 'ATD > 240 min'),
    (480, None, 'ATD > 480 min'),
]:
    if lo is None:
        mask = df['ATD'] <= hi
    elif hi is None:
        mask = df['ATD'] > lo
    else:
        mask = (df['ATD'] > lo) & (df['ATD'] <= hi)
    n = mask.sum()
    print(f'  {label:<30} : {n:>7,}  ({n/len(df)*100:.3f}%)')

In [ ]:
# Flag outliers before removing
atd_p99 = df['ATD'].quantile(0.99)
atd_p01 = df['ATD'].quantile(0.01)

df['is_atd_outlier'] = ((df['ATD'] <= 0) | (df['ATD'] > atd_p99)).astype(int)
print(f'ATD p01 = {atd_p01:.2f} min  |  p99 = {atd_p99:.2f} min')
print(f'Flagged as outlier: {df["is_atd_outlier"].sum():,} rows ({df["is_atd_outlier"].mean()*100:.2f}%)')

# Create modeling-ready ATD: remove impossible, cap extreme
df_model = df[df['ATD'] > 0].copy()
df_model['ATD_capped'] = df_model['ATD'].clip(upper=atd_p99)

print(f'\nRows after removing ATD ≤ 0: {len(df_model):,}')
print(f'ATD_capped stats:')
print(df_model['ATD_capped'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Before capping
axes[0].hist(df_model['ATD'], bins=100, color='gray', edgecolor='none')
axes[0].set_yscale('log')
axes[0].set_title('ATD — Raw (log y-scale)')
axes[0].set_xlabel('Minutes')

# After capping
axes[1].hist(df_model['ATD_capped'], bins=80, color='steelblue', edgecolor='none')
axes[1].axvline(df_model['ATD_capped'].median(), color='red', linestyle='--',
                label=f'Median {df_model["ATD_capped"].median():.0f} min')
axes[1].legend()
axes[1].set_title(f'ATD — Capped at p99 ({atd_p99:.0f} min)')
axes[1].set_xlabel('Minutes')

# Outliers by segment
out_by_territory = (
    df.groupby('territory', observed=True)['is_atd_outlier']
    .mean() * 100
).sort_values(ascending=False)
out_by_territory.plot(kind='bar', ax=axes[2], color='tomato', edgecolor='none')
axes[2].set_title('ATD Outlier Rate by Territory (%)')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=35)

plt.suptitle('Outlier Treatment', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Categorical Encoding

We create numeric codes for model compatibility (LightGBM can also consume categories directly, but we prepare both).

In [ ]:
cat_cols = ['territory', 'courier_flow', 'geo_archetype', 'merchant_surface', 'time_block', 'day_type']

for col in cat_cols:
    if df_model[col].dtype.name != 'category':
        df_model[col] = df_model[col].astype('category')
    df_model[f'{col}_code'] = df_model[col].cat.codes

# Summary of category cardinality
cat_summary = pd.DataFrame({
    'n_categories': {c: df_model[c].nunique() for c in cat_cols},
    'top_value':    {c: df_model[c].value_counts().index[0] for c in cat_cols},
    'top_pct':      {c: f"{df_model[c].value_counts().iloc[0]/len(df_model)*100:.1f}%" for c in cat_cols},
})
display(cat_summary)

In [ ]:
# Target encoding: segment median ATD per territory (using full set; for prod use CV folds)
territory_median_atd = df_model.groupby('territory', observed=True)['ATD_capped'].median()
df_model['territory_median_atd'] = df_model['territory'].map(territory_median_atd)

geo_median_atd = df_model.groupby('geo_archetype', observed=True)['ATD_capped'].median()
df_model['geo_archetype_median_atd'] = df_model['geo_archetype'].map(geo_median_atd)

print('Target encoding complete (territory, geo_archetype).')
display(territory_median_atd.sort_values(ascending=False).rename('median_ATD (min)').round(1).to_frame())

---
## 9. Feature Summary & Save

In [ ]:
FEATURE_COLS = [
    # Time
    'hour_utc', 'hour_local', 'day_of_week', 'is_weekend', 'is_peak_hour',
    'time_block_code', 'day_type_code', 'week_number', 'month',
    # Distance (raw + transformed)
    'pickup_distance', 'dropoff_distance', 'total_distance', 'distance_ratio',
    'log_pickup', 'log_dropoff', 'log_total_dist', 'is_long_trip', 'is_distance_missing',
    # Wait time
    'eater_wait_min',
    # Categoricals (encoded)
    'territory_code', 'courier_flow_code', 'geo_archetype_code', 'merchant_surface_code',
    # Target encoded
    'territory_median_atd', 'geo_archetype_median_atd',
]

TARGET_COLS = ['ATD', 'ATD_capped', 'is_atd_outlier']
ID_COLS     = ['workflow_uuid', 'driver_uuid', 'delivery_trip_uuid', 'restaurant_offered_timestamp_utc']

df_save = df_model[ID_COLS + FEATURE_COLS + TARGET_COLS + cat_cols].copy()

print(f'Feature dataset shape: {df_save.shape}')
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
df_save.to_parquet(FEATURES_PATH, index=False)
print(f'Saved → {FEATURES_PATH}  ({FEATURES_PATH.stat().st_size / 1024**2:.1f} MB)')

# Feature completeness report
missing_report = df_save[FEATURE_COLS].isnull().sum().to_frame('missing')
missing_report['pct'] = (missing_report['missing'] / len(df_save) * 100).round(3)
print('\nFeature completeness (all should be 0):')
display(missing_report[missing_report['missing'] > 0])
if (missing_report['missing'] == 0).all():
    print('✓ All feature columns are 100% complete.')

In [ ]:
# Final feature correlation with ATD_capped
numeric_features = [
    'hour_local', 'day_of_week', 'is_weekend', 'is_peak_hour',
    'pickup_distance', 'dropoff_distance', 'total_distance', 'distance_ratio',
    'log_total_dist', 'eater_wait_min',
    'territory_median_atd', 'geo_archetype_median_atd',
]

corr_final = (
    df_save[numeric_features + ['ATD_capped']]
    .corr()['ATD_capped']
    .drop('ATD_capped')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_final.values]
corr_final.plot(kind='bar', ax=ax, color=colors, edgecolor='none')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with ATD_capped (Pearson r)', fontweight='bold')
ax.set_ylabel('Pearson r')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

print('\nTop features by absolute correlation:')
display(corr_final.to_frame('pearson_r').round(4))